# Spark training
---
## Getting Started
We first import necessary modules and create a new `SparkSession`.

Don't import functions like this
```
# bad
from pyspark.sql.functions import *

# not recommended, sum is also a Python function
from pyspark.sql.functions import sum, avg, when
```

Instead do this
```
# good
from pyspark.sql import functions as F
````

Similarly with importing types

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('SparkTraining')
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider')
    .getOrCreate()
)

Here we read the raw data from S3 with parquet file format, using the configuration we defined earlier in the `SparkSession`.

In [0]:
S3_PATH = 's3a://aws-bigdata-blog/generated_synthetic_reviews/data/'

df_bronze = spark.read.parquet(S3_PATH)

To learn about any Python function, we can use `?`. Try to run the following and read about `printSchema`.

In [0]:
df_bronze.printSchema?

Now run the function.

In [0]:
df_bronze.printSchema()

---
## Actions
Spark is divided into transformations and actions.

A **transformation** defines a new dataframe. Unlike pandas (or SQL), PySpark transformations are "lazy".

Unlike a transformation, an **action** is executed immediately.

Spark builds a DAG/logical plan of all the transformations, and executes the plan when an action is triggered.

The 3 main actions are `count`, `show`, and `collect` (which we **never** want to use).

Feel free to add code cells and use `?` to read about other things, such as `count`.

In [0]:
df_bronze.count()

PySpark's native way to see the dataframe is with `.show()`. However in Databricks we can also use `display()` which prints in a prettier way.

In [0]:
df_bronze.show()

In [0]:
display(df_bronze)

---
## Transformations
- Read about `F.col`, `select`, `withColumnRenamed`.
- Grab just the product ID, title, rating and the review's date.
- Rename the star rating column to just "rating".
- Show the "slim" dataframe.

In [0]:
# change this
df_slim = df_bronze

display(df_slim)

- Read about `filter` (same as `where`) and bitwise operators and `isin`.
- A "superb" review is a review with 4/5 stars OR at least 20 helpful votes.
- Show only superb reviews.

In [0]:
# change this
df_superb = df_bronze

display(df_superb)

- Read about `withColumn`.
- We want to the ratio of helpful votes divided by total votes.
- Create a new column called "helpful_ratio" that calculates this metric.
- Show result (if it fails - read the error message).

In [0]:
# change this
df_ratio = df_bronze

display(df_ratio)

- Read about `groupBy`, `orderBy`, `agg`, `F.avg`, `F.count`.
- For each category, calculate the average rating as "avg_rating" and the total number of reviews as "review_count", and sort the list so the highest average ratings are at the top.
- Show the result.

In [0]:
def category_metrics(df_bronze):
    # change this
    return df_bronze

df_silver = category_metrics(df_bronze)
display(df_silver)

The BI team wants to put this on an executive dashboard, but the current format is too messy for a chart. They've handed you a list of business rules to format the final dataset.

- Read about `F.regexp_replace`, `F.when` (and its partner `.otherwise()`), `F.round`, `F.concat`, `F.lit`, `df.drop`.

The Business Requirements:
- The executives hate underscores. Convert categories such as `"Grocery_Gourmet_Food"` into `"Grocery Gourmet Food"`.
- Round the average rating column to exactly 2 decimal places.
- Create a new column called `volume_tier`. If a category has 8,000,000 or more reviews, label it `"High"`. If it has 5,000,000 or more, label it `"Medium"`. Anything else is `"Low"`.
- Dashboards need short numbers. Transform the `review_count` into a new column called `reviews_in_millions` that formats the number into a string like `"9.67M"`.
- Drop the number of reviews column so the BI tool doesn't accidentally use it.
- The executives only care about the big players. Filter out any category in the `"Low"` tier.
- Order the final dataset so the highest-rated categories are at the top.

In [0]:
def category_dashboard(df_silver):
    # change this
    return df_silver

df_gold = category_dashboard(df_silver)
display(df_gold)

---
# Spark SQL

So far, we worked with the dataframe API. Spark has a SQL API as well.

In order to use the dataframe in SQL, we first need to register it as a temporary view.

In [0]:
df_bronze.createOrReplaceTempView('src_reviews')

select_query = """
SELECT *
FROM src_reviews
"""

display(spark.sql(select_query))

Try to implement the BI requirements in Spark SQL to get the dashboard-ready dataframe.

Note that we can also combine the two - dataframe API + Spark SQL.

In [0]:
df_silver.createOrReplaceTempView('reviews_categories')

# change this
gold_query = """
WITH dw_reviews AS (
    SELECT 
        *
    FROM reviews_categories
    )
SELECT 
    product_category,
    avg_rating,
    volume_tier,
    reviews_in_millions
FROM dw_reviews
WHERE volume_tier != 'Low'
ORDER BY avg_rating DESC
"""

df_gold_sql = spark.sql(gold_query)

display(df_gold_sql)

---
## Testing

We can check for diffs between the two approaches.

In [0]:
def diffs(df1, df2):
    return df1.exceptAll(df2).union(df2.exceptAll(df1))

diffs = diffs(df_gold, df_gold_sql)
print(diffs.count())

PySpark has a testing function called `assertDataFrameEqual` which checks both the data and schema match, whereas the `diffs` function we wrote only checks for data match (on the same column names). Use `assertDataFrameEqual` because it will actually check for diffs, and raise an AssertionError if found.

In [0]:
# this setting is only for databricks
import logging
logging.getLogger("pyspark.databricks.pandas.usage_logger").setLevel(logging.CRITICAL)

from pyspark.testing import assertDataFrameEqual
assertDataFrameEqual(df_gold, df_gold_sql)

If you solved correctly - you should see nothing here.

Now rename the `product_category` column name to `category`, and see what happens.

In [0]:
def rename_category(df_gold):
  # change this
  return df_gold

df_gold_renamed = rename_category(df_gold)
assertDataFrameEqual(df_gold, df_gold_renamed)

---
## Window Functions
Similar to SQL. A nice feature is that we can reuse the same window variable.

- Read about `Window.partitionBy`.
- For each volume tier, we want to create columns `tier_max` and `tier_min` with the highest and lowest average rating **in that tier**.

In [0]:
# change this
def analyze_volume_tiers(df):
    return df

df_analyzed = analyze_volume_tiers(df_gold)
display(df_analyzed)

---
## `cache`, `persist`, `collect`
`collect` (also `toPandas`) loads a dataframe into the driver's memory. As we mentioned, we want to avoid this, because it cancels Spark's nature of distributed computing and might cause OOM (out of memory) errors.

It is useful only when we must use perform something that cannot be distributed by Spark, such as algorithms or visualizations in matplotlib.

Let's try and collect `df_analyzed` using `toPandas`. It should fit in memory if you solved correctly.

In [0]:
import matplotlib.pyplot as plt

pdf = df_analyzed.toPandas().sort_values("volume_tier")
pdf.plot(x='product_category', y=['tier_max', 'tier_min'], kind='bar')
plt.tight_layout()
plt.show()

Read about `cache` and `persist`. Similar to `collect`, we don't want to use `persist`.

Go back to the previous exercises and cache `dw_silver`, then runs the next cells and it should run faster.

---
## Functions
As you already saw, we can write PySpark transformations as Python functions for reuse.

Write a function that takes a (PySpark) dataframe and returns a dataframe with a year column, containing each year in the dataset **only once**.

In [0]:
def get_years(df):
    # change this
    return df

# test it - should see each year only once
get_years(df_bronze).select('review_year').orderBy('review_year', ascending=False).show()

Okay - let's try and write a function that checks if the year is before 2000.

### Attempt #1

In [0]:
def check_year1(year):
    return year < 2000

before_2000_attempt1 = df_bronze.filter(check_year1(F.col('review_year')))
get_years(before_2000_attempt1).orderBy('review_year').show()

It worked. Now let's tweak this a little.

### Attempt #2

In [0]:
def check_year2(year):
    if year < 2000:
        return True
    return False

before_2000_attempt2 = df_bronze.filter(check_year2(F.col('review_year')))
get_years(before_2000_attempt2).orderBy('review_year').show()

This failed. Note that `F.col('review_year')` is a column and not the actual row data, so Python doesn't know how to evaluate it as True or False.

Then how did Attempt #1 work?

Well, a small illusion. What we passed to `check_year1` was the column, so the `<` operator that was used was actually PySpark's.

Is there a way to overcome this?

Yes, by registering a UDF (user defined function).

### Attempt #3

In [0]:
@F.udf(returnType=T.BooleanType())
def check_year3(year):
    if year < 2000:
        return True
    return False

before_2000_attempt3 = df_bronze.filter(check_year3(F.col('review_year')))
get_years(before_2000_attempt3).orderBy('review_year').show()

This worked but it's slower and not efficient. **Don't use UDFs**. Rely on `F` (functions).

---

## Query Plans

Spark builds a DAG and executes it when an action is called. We can use `.explain()` to read the execution plan.

A plan can be read from bottom to top, and it allows us to see how Spark translates our code into cluster operations.

In [0]:
before_2000_attempt1.explain()

In [0]:
before_2000_attempt3.explain()

Comparing the two plans, you can see that Attempt #1 filters the year using DataFilters, whereas Attempt #3 uses BatchEvalPython. **This is why UDFs are bad**. Spark runs on the Scala JVM, whenever it needs to call Python, it can't perform the necessary optimizations, it treats the Python function as a "black box".

---
## Predicate Pushdown

Spark optimizes plans by pushing filters down to the storage layer, so that less data is processed.

In [0]:
df_no_insight = df_bronze.where(F.col('insight')=='N').select('insight','star_rating').orderBy('star_rating')
display(df_no_insight)
df_no_insight.explain()

Look closely at the `PhotonScan parquet` step at the very bottom. Notice this section:
`DataFilters: [isnotnull(insight), (insight = N)]`

This is **Predicate Pushdown**. Instead of loading the entire massive dataset from S3 into Spark's memory and *then* applying the `WHERE` clause, Spark "pushes down" the filter directly to the storage layer. 

Because Parquet is an optimized columnar format, Spark uses these filters to skip reading entire blocks of irrelevant data from disk. This makes your query run much faster.

---
## Joins
PySpark dataframes are joined just like SQL. But Spark also has several kinds of joins - mainly **Sort Merge Join** and **Broadcast Join**.

In [0]:
!echo $(pwd)/countries.csv

Copy this file path to the next cell

In [0]:
# change this
COUNTRIES_CSV = "countries.csv"

df_countries = spark.read.csv(COUNTRIES_CSV, header=True).select(
        F.col("alpha-2").alias("iso"),
        F.col("name").alias("country"),
        F.col("region")
    )

display(df_countries)

We will perform a LEFT JOIN with the countries dataframe. We will hint Spark to use Sort Merge Join.

In [0]:
df_merge = df_bronze.join(
    df_countries.hint('merge'),
    on=F.col('marketplace') == F.col('iso'),
    how='left',
    )
df_merge.printSchema()

Let's say we want to count the average rating per region, and sort according to the rating average.

In [0]:
# change this
def rating_by_region(df):
    return df

df_rating_by_region = rating_by_region(df_merge)
df_rating_by_region.show()

Look at the following plan. You should see Spark uses SortMergeJoin here.

In [0]:
df_rating_by_region.explain()

Now let's use Broadcast Join.

In [0]:
df_broadcast = df_bronze.join(
    df_countries.hint('broadcast'),
    on=F.col('marketplace') == F.col('iso'),
    how='left',
    )
df_broadcast.printSchema()

In [0]:
df_rating_by_region_broadcast = rating_by_region(df_broadcast)
df_rating_by_region_broadcast.show()

Using broadcast here seems to be faster than merge. But why?

Spark spreads data across many workers.

When you join dataframes, it must move data between workers in order to match keys.

This is called a **shuffle** - and it is the biggest performance killer in Spark.

A broadcast in Spark sends a complete copy of the dataframe to every worker.

That way there's no shuffle involved when joining it.

This works best when the dataframe we broadcast is small, so it fits in the worker's memory.

In [0]:
df_rating_by_region_broadcast.explain()

As you can see, this plan uses BroadcastHashJoin.

Check which join strategy is used without any hint.

---
## Partition Pruning
When data is saved to S3 or a database, we often organize it into folders based on a specific column.

If your data is partitioned on disk by a column, and you run a query filtering values in this column, Spark will skip all the other folders.

Let's see that in action. Look at the explain plans of the following two filters, try to find what the partition column is.

In [0]:
df_star_filter = df_bronze.filter("star_rating == 5")
df_category_filter = df_bronze.filter("product_category == 'Apparel'")

In [0]:
df_star_filter.explain()

In [0]:
df_category_filter.explain()

---
## Writing Data
When saving data, you must tell Spark what to do if a file already exists in that location. We do this using `.mode()`:
*   **`overwrite`**: Deletes the existing data and replaces it with the new data.
*   **`append`**: Adds the new data to the existing data.

Let's try this.

- Count the number of rows in df_gold.
- Write df_gold to a folder in parquet format using overwrite mode.
- Then do it again using append.
- Read from the folder and count the number of rows.

In [0]:
# TODO